# Dialforge Sales Hard Acceptance v3.4 — Kaggle

This gate measures the **actual shipping architecture**. Raw Qwen sales intelligence must still score at least 85, while deterministic call-control actions are evaluated through Dialforge's production-equivalent router. Raw native Ollama tool-call accuracy remains visible as a diagnostic.

Marketing passes only when every shipping tier has **overall >=85, raw sales >=85, production tool routing >=90, and zero critical spoken hallucinations**.

**Before Run All:**
1. Kaggle Settings → Accelerator → NVIDIA GPU. One T4 is enough.
2. Turn **Internet ON**.
3. Click **Run All**. If this is the same Kaggle session, existing Ollama/model downloads are reused.


In [ ]:
import os, pathlib, shutil, subprocess, sys, time, json
ROOT = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'
OUT = ROOT / 'dialforge-hard-sales-v3_4'
print('=== DIALFORGE KAGGLE PRODUCTION SALES ACCEPTANCE v3.4 ===')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Enable a Kaggle GPU accelerator.')
print('GPU(s):\n' + gpu.stdout.strip())
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'
probe = subprocess.run(['curl', '-I', '-L', '--max-time', '15', 'https://github.com'], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Internet appears disabled. Turn Internet ON in Kaggle settings and rerun.')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned source:', sha)
if shutil.which('zstd') is None:
    subprocess.run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd', shell=True, check=True)
if shutil.which('ollama') is None:
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
import requests
def ollama_ready():
    try:
        return requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok
    except Exception:
        return False
if ollama_ready():
    print('Reusing existing Ollama server and model cache.')
else:
    log_path = ROOT / 'ollama-hard-sales-v3_4.log'
    log = open(log_path, 'w')
    ollama = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(90):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        log.flush()
        raise RuntimeError('Ollama did not become ready. Log: ' + str(log_path))
print('Ollama ready. Running production-equivalent v3.4 gate...')


In [ ]:
import subprocess, json, pathlib, pandas as pd, os, sys
OUT.mkdir(parents=True, exist_ok=True)
runner = REPO / 'benchmarks' / 'dialforge_sales_hard_acceptance_v3_4.py'
cmd = [sys.executable, str(runner), '--output-dir', str(OUT)]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
lines = []
for line in proc.stdout:
    print(line, end='')
    lines.append(line)
code = proc.wait()
(OUT / 'kaggle-run.log').write_text(''.join(lines), encoding='utf-8')
report_path = OUT / 'dialforge-sales-hard-v3.json'
if not report_path.exists():
    raise RuntimeError(f'No report was produced. Exit code {code}. Check {OUT / "kaggle-run.log"}')
report = json.loads(report_path.read_text(encoding='utf-8'))
rows = []
for model, data in report['models'].items():
    sales = data['sales']
    tools = data['tools']
    passed = (float(data['score']) >= 85.0 and float(sales.get('raw_score', sales['score'])) >= 85.0 and float(tools['accuracy']) >= 90.0 and int(sales['critical_failures']) == 0)
    rows.append({
        'Model':model,
        'Overall':data['score'],
        'Product Sales':sales['score'],
        'Raw Sales':sales.get('raw_score'),
        'Product Tools':tools['accuracy'],
        'Raw Native Tools':tools.get('raw_accuracy'),
        'Integrity':sales['integrity'],
        'Product Critical':sales['critical_failures'],
        'Raw Critical':sales.get('raw_critical_failures'),
        'PASS':passed,
    })
display(pd.DataFrame(rows))

print('\nFAILED PRODUCTION TOOL CASES')
tool_rows=[]
for model,data in report['models'].items():
    for case in data['tools']['cases']:
        if not case['passed']:
            tool_rows.append({'Model':model,'Case':case['id'],'Expected':case['expected'],'Tools':', '.join(case['tools']),'Source':case.get('source'),'Content':case['content']})
if tool_rows: display(pd.DataFrame(tool_rows))
else: print('None')

print('\nRAW NATIVE TOOL FAILURES (diagnostic only)')
raw_tool_rows=[]
for model,data in report['models'].items():
    for case in data['tools'].get('raw_cases', []):
        if not case['passed']:
            raw_tool_rows.append({'Model':model,'Case':case['id'],'Expected':case['expected'],'Tools':', '.join(case['tools']),'Content':case['content']})
if raw_tool_rows: display(pd.DataFrame(raw_tool_rows))
else: print('None')

print('\nWEAK OR GUARDED SALES TURNS')
weak=[]
for model,data in report['models'].items():
    for conv in data['sales']['conversations']:
        for turn in conv['turns']:
            if float(turn['score']) < 85 or int(turn['critical']) > 0 or turn.get('raw_answer') != turn.get('answer'):
                weak.append({
                    'Model':model,'Conversation':conv['id'],'Expected':turn['expect'],
                    'Product Score':turn['score'],'Raw Score':turn.get('raw_score'),
                    'Failures':', '.join(turn['failures']),
                    'Product Answer':turn['answer'],'Raw Answer':turn.get('raw_answer')
                })
if weak: display(pd.DataFrame(weak))
else: print('None')

ready = all(
    float(data['score']) >= 85.0 and
    float(data['sales'].get('raw_score', data['sales']['score'])) >= 85.0 and
    float(data['tools']['accuracy']) >= 90.0 and
    int(data['sales']['critical_failures']) == 0
    for data in report['models'].values()
)
print('\n' + '='*72)
print('MARKETING GATE:', 'PASS' if ready else 'BLOCKED')
print('='*72)
print('Report:', report_path)
print('Log:', OUT / 'kaggle-run.log')
